In [17]:
import os
import pandas as pd
import xml.etree.ElementTree as ET

# 파일 경로
senario_path = r"C:\Users\(주)내일이엔시 도로교통안전연구소\Desktop\프로젝트\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\1. 회의자료\26.05.18\synthetic_scenarios_all_140_with_set_type(영향권5(진출))_3차로폐쇄.csv"
inpx_path = r"E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\시나리오(140-유고지점 Uniform Random)_260518\진입구간\화성~서울(140-유고지점)_260526.inpx"

save_path = r"E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\시나리오(140-유고지점 Uniform Random)_260518"

# 시나리오 / inpx 로드
senario_df = pd.read_csv(senario_path, encoding="cp949")
tree = ET.parse(inpx_path)
root = tree.getroot()

# 3차로 검지기 목록
three_lane = [71, 72, 73, 119, 120, 125, 126, 188, 189, 216, 217, 218, 219, 220]

# 검지기 번호 → 링크 번호 매핑
detector_map = {}

for dcp in root.findall(".//dataCollectionPoint"):
    detector_no = int(dcp.get("no")) % 1000
    detector_link = int(dcp.get("lane").split()[0])
    detector_map[detector_no] = detector_link


def get_upstream_detector(root, location):
    detector_no = None
    detector_link = None
    gradient = None

    rsa_lane = None
    rsa_pos = None
    best_pos = -1

    # 시나리오의 유고 위치명과 inpx의 reducedSpeedArea name 매칭
    for rsa in root.findall(".//reducedSpeedArea"):
        if rsa.get("name") == location:

            rsa_lane = rsa.get("lane")
            rsa_pos = float(rsa.get("pos"))

            # 같은 링크/차로에서 유고 지점보다 상류에 있는 가장 가까운 검지기 탐색
            for dcp in root.findall(".//dataCollectionPoint"):
                if dcp.get("lane") == rsa_lane:
                    dcp_pos = float(dcp.get("pos"))

                    if rsa_pos >= dcp_pos >= best_pos:
                        best_pos = dcp_pos

                        # 검지기 번호 가공
                        detector_no = int(dcp.get("no")) % 1000 - 1
                        detector_link = detector_map.get(detector_no)

    # 같은 링크에서 상류 검지기를 못 찾은 경우, 하류 방향 보정
    if detector_no is None:
        best_pos = float("inf")

        for dcp in root.findall(".//dataCollectionPoint"):
            if dcp.get("lane") == rsa_lane:
                dcp_pos = float(dcp.get("pos"))

                if rsa_pos < dcp_pos < best_pos:
                    best_pos = dcp_pos
                    detector_no = int(dcp.get("no")) % 1000 - 1 - 1
                    detector_link = detector_map.get(detector_no)

    # 검지기가 위치한 링크의 종단경사 추출
    for link in root.findall(".//link"):
        if int(link.get("no")) == detector_link:
            gradient = round(float(link.get("gradient")), 4)
            break

    # 본선 차로 수 판단
    if detector_no in three_lane:
        main_lane = 3
    else:
        main_lane = 2

    return detector_no, detector_link, gradient, main_lane


# 시나리오별 검지기 및 종단경사 추출
gradient_list = []

for idx, row in senario_df.iterrows():
    incident_location = row["location_name"]

    # location_name에 여러 위치가 들어있는 경우 첫 번째 위치 사용
    locations = [x.strip() for x in incident_location.split(",")]
    location = locations[0]

    detector_no, detector_link, gradient, main_lane = get_upstream_detector(root, location)

    gradient_list.append({
        "시나리오번호": idx + 1,
        "유고위치": location,
        "검지기번호": detector_no,
        "검지기링크": detector_link,
        "종단경사": gradient,
        "본선차로수": main_lane
    })

gradient_df = pd.DataFrame(gradient_list)
gradient_df.to_excel(os.path.join(save_path,"영향권5 종단경사.xlsx"), index=True)
print(gradient_df)

     시나리오번호         유고위치  검지기번호  검지기링크   종단경사  본선차로수
0         1  영향권5(진출)1-1    214     33 -0.002      2
1         2  영향권5(진출)5-1    218     37 -0.002      3
2         3  영향권5(진출)2-1    215     33 -0.002      2
3         4  영향권5(진출)5-1    218     37 -0.002      3
4         5  영향권5(진출)5-1    218     37 -0.002      3
..      ...          ...    ...    ...    ...    ...
105     106  영향권5(진출)3-1    216     37 -0.002      3
106     107  영향권5(진출)1-1    214     33 -0.002      2
107     108  영향권5(진출)4-1    217     37 -0.002      3
108     109  영향권5(진출)2-1    215     33 -0.002      2
109     110  영향권5(진출)4-1    217     37 -0.002      3

[110 rows x 6 columns]
